In [ ]:
# 1. 安装 Kaggle CLI
!pip install kaggle -q

# 2. 上传你的 kaggle.json（从 Kaggle → Account → Create API Token 下载）
from google.colab import files
files.upload()  # 选择你的 kaggle.json

# 3. 配置权限
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# 4. 下载数据集
!kaggle datasets download -d shubhamkarande13/d-fire

# 5. 解压到指定文件夹
!unzip d-fire.zip -d /content/D-Fire

In [ ]:
!pip install compressai
!pip install ultralytics

In [ ]:
import glob
import os
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.optim as optim
from PIL import Image
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import transforms

try:
    from compressai.models import Cheng2020Anchor
except ImportError:
    raise ImportError("请先运行: pip install compressai")

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(iterable, **kwargs):
        return iterable

# ===== 配置（与你原来保持一致）=====
IMG_SIZE        = 384
BATCH_SIZE      = 16
NUM_WORKERS     = 0
MAX_TRAIN_SAMPLES = 2048
MAX_VAL_SAMPLES   = 512
LOG_EVERY       = 20
DATA_ROOT       = "/content/D-Fire"
EPOCHS          = 30
EARLY_STOP_PATIENCE = 5

# 率失真权衡：λ 越大越重视失真（PSNR），越小越重视压缩率（bpp）
# 推荐先用 0.01 跑通，再试 0.001（更低比特率）或 0.05（更高PSNR）
LAMBDA          = 0.05

LR_MAIN         = 5e-5   # 主网络（encoder + decoder + hyperprior）
LR_AUX          = 1e-3   # 熵模型辅助参数，CompressAI 官方推荐比主网络大一个量级
GRAD_CLIP_NORM  = 1.0

save_dir = "outputs_compressai"
os.makedirs(save_dir, exist_ok=True)

history = {
    "train_loss": [], "val_loss": [],
    "train_psnr": [], "val_psnr": [],
    "train_bpp":  [], "val_bpp":  [],
}

# ===== 设备 =====
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    torch.backends.cudnn.benchmark = True
print(f"Device: {device}", flush=True)
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}", flush=True)


# ===== 数据集（完全复用你原来的 DFireDataset）=====
# 注意：去掉了 Normalize，CompressAI 期望输入在 [0, 1]
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.Lambda(lambda img: img.convert("RGB") if img.mode != "RGB" else img),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),          # 输出已在 [0, 1]，不再 Normalize
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.Lambda(lambda img: img.convert("RGB") if img.mode != "RGB" else img),
    transforms.ToTensor(),
])


class DFireDataset(Dataset):
    """D-Fire: root/{train,test}/images/*.jpg  （原样复用）"""
    def __init__(self, root, split="train", transform=None):
        assert split in ("train", "test")
        img_dir = os.path.join(root, split, "images")
        self.paths = sorted(
            glob.glob(os.path.join(img_dir, "*.jpg"))
            + glob.glob(os.path.join(img_dir, "*.png"))
        )
        if not self.paths:
            raise FileNotFoundError(f"在 {img_dir} 下找不到图片")
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, 0


def maybe_subset(dataset, max_samples):
    if max_samples is None or max_samples >= len(dataset):
        return dataset
    return Subset(dataset, list(range(max_samples)))


@torch.no_grad()
def batch_psnr(recon, target):
    """原样复用"""
    mse = (recon - target).pow(2).mean(dim=(1, 2, 3))
    return (10 * torch.log10(1.0 / mse.clamp(min=1e-8))).mean().item()


print("Loading dataset...", flush=True)
train_db = maybe_subset(
    DFireDataset(DATA_ROOT, split="train", transform=train_transform),
    MAX_TRAIN_SAMPLES,
)
val_db = maybe_subset(
    DFireDataset(DATA_ROOT, split="test", transform=val_transform),
    MAX_VAL_SAMPLES,
)

pin_memory = device.type == "cuda"
train_loader = DataLoader(train_db, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=pin_memory)
val_loader   = DataLoader(val_db,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=pin_memory)

print(
    f"Train: {len(train_db)} imgs, {len(train_loader)} batches | "
    f"Val: {len(val_db)} imgs, {len(val_loader)} batches | "
    f"{IMG_SIZE}x{IMG_SIZE} bs={BATCH_SIZE}",
    flush=True,
)

# ===== 模型 =====
# quality=3 对应中等比特率，范围 1~8；T4 显存跑 quality<=5 比较稳
# pretrained=False 从头在 D-Fire 上训练
print("Building model...", flush=True)
# N=128 是超先验通道数，M=192 是主潜变量通道数
# 对应原来 quality=3 的参数量，T4 显存完全够用
model = Cheng2020Anchor(N=128).to(device)
print(f"模型参数量: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M", flush=True)

# ===== 优化器（CompressAI 标准双优化器写法）=====
# main optimizer: encoder + decoder + hyperprior 主体参数
# aux  optimizer: 熵模型内部的 CDF 参数，单独更新
optimizer     = optim.Adam(model.parameters(), lr=LR_MAIN)
aux_optimizer = optim.Adam(model.entropy_bottleneck.parameters(), lr=LR_AUX)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=2, min_lr=1e-7
)

# ===== 训练循环 =====
best_psnr      = 0.0
epochs_no_improve = 0

for epoch in range(EPOCHS):
    t_epoch = time.time()

    # ---------- Train ----------
    model.train()
    total_loss = total_bpp = total_psnr = 0.0
    n_batches = 0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [train]", leave=False)
    for step, (imgs, _) in enumerate(pbar, 1):
        imgs = imgs.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        aux_optimizer.zero_grad(set_to_none=True)

        # forward：返回 {"x_hat", "likelihoods": {"y":…, "z":…}}
        out = model(imgs)
        x_hat = out["x_hat"].clamp(0, 1)

        # 比特率估算（bpp）
        num_pixels = imgs.shape[0] * imgs.shape[2] * imgs.shape[3]
        bpp = sum(
            (-torch.log2(lk).sum() / num_pixels)
            for lk in out["likelihoods"].values()
        )

        # 失真（MSE，与你原来保持一致；若要换 MS-SSIM 在此处替换）
        distortion = torch.nn.functional.mse_loss(x_hat, imgs)

        # 率失真联合损失
        loss = bpp + LAMBDA * 255**2 * distortion
        # 注：乘以 255^2 是将 MSE 从 [0,1] 域折算到 [0,255] 域，
        #     让 LAMBDA 的量级和文献中保持一致

        loss.backward()
        if GRAD_CLIP_NORM:
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
        optimizer.step()

        # 辅助损失（熵模型 CDF 参数），独立 backward + step
        aux_loss = model.entropy_bottleneck.loss()
        aux_loss.backward()
        aux_optimizer.step()

        total_loss += loss.item()
        total_bpp  += bpp.item()
        total_psnr += batch_psnr(x_hat, imgs)
        n_batches  += 1

        if step % LOG_EVERY == 0:
            pbar.set_postfix(loss=f"{loss.item():.4f}", bpp=f"{bpp.item():.3f}")

    avg_train_loss = total_loss / n_batches
    avg_train_bpp  = total_bpp  / n_batches
    avg_train_psnr = total_psnr / n_batches

    # ---------- Validation ----------
    model.eval()
    val_loss = val_bpp = val_psnr = 0.0
    n_val = 0

    with torch.no_grad():
        for imgs, _ in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [val]", leave=False):
            imgs = imgs.to(device, non_blocking=True)
            out  = model(imgs)
            x_hat = out["x_hat"].clamp(0, 1)

            num_pixels = imgs.shape[0] * imgs.shape[2] * imgs.shape[3]
            bpp = sum(
                (-torch.log2(lk).sum() / num_pixels)
                for lk in out["likelihoods"].values()
            )
            distortion = torch.nn.functional.mse_loss(x_hat, imgs)
            loss = bpp + LAMBDA * 255**2 * distortion

            val_loss += loss.item()
            val_bpp  += bpp.item()
            val_psnr += batch_psnr(x_hat, imgs)
            n_val    += 1

    avg_val_loss = val_loss / n_val
    avg_val_bpp  = val_bpp  / n_val
    avg_val_psnr = val_psnr / n_val
    elapsed = time.time() - t_epoch

    history["train_loss"].append(avg_train_loss)
    history["val_loss"].append(avg_val_loss)
    history["train_psnr"].append(avg_train_psnr)
    history["val_psnr"].append(avg_val_psnr)
    history["train_bpp"].append(avg_train_bpp)
    history["val_bpp"].append(avg_val_bpp)

    scheduler.step(avg_val_psnr)

    print(
        f"Epoch {epoch+1}/{EPOCHS} ({elapsed:.1f}s) | "
        f"Train Loss {avg_train_loss:.4f} | Val Loss {avg_val_loss:.4f} | "
        f"Train PSNR {avg_train_psnr:.2f} | Val PSNR {avg_val_psnr:.2f} dB | "
        f"Val bpp {avg_val_bpp:.4f}",
        flush=True,
    )

    # ---------- Checkpoint ----------
    if avg_val_psnr > best_psnr:
        best_psnr = avg_val_psnr
        epochs_no_improve = 0
        torch.save(
            {
                "epoch": epoch + 1,
                "model": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "val_psnr": avg_val_psnr,
                "val_bpp":  avg_val_bpp,
            },
            f"{save_dir}/best_model.pth",
        )
        print(f"  → 保存最优模型 PSNR={best_psnr:.2f} dB  bpp={avg_val_bpp:.4f}", flush=True)
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= EARLY_STOP_PATIENCE:
            print(
                f"Early stop: Val PSNR 连续 {EARLY_STOP_PATIENCE} epoch "
                f"未超过 {best_psnr:.2f} dB",
                flush=True,
            )
            break

# ===== 曲线绘图（原样复用）=====
epochs_ran = range(1, len(history["train_loss"]) + 1)

plt.figure()
plt.plot(epochs_ran, history["train_loss"], label="Train Loss")
plt.plot(epochs_ran, history["val_loss"],   label="Val Loss")
plt.xlabel("Epoch"); plt.ylabel("Loss")
plt.title("Loss Curve"); plt.legend(); plt.grid()
plt.savefig(f"{save_dir}/loss_curve.png"); plt.show()

plt.figure()
plt.plot(epochs_ran, history["train_psnr"], label="Train PSNR")
plt.plot(epochs_ran, history["val_psnr"],   label="Val PSNR")
plt.xlabel("Epoch"); plt.ylabel("PSNR (dB)")
plt.title("PSNR Curve"); plt.legend(); plt.grid()
plt.savefig(f"{save_dir}/psnr_curve.png"); plt.show()

plt.figure()
plt.plot(epochs_ran, history["train_bpp"], label="Train bpp")
plt.plot(epochs_ran, history["val_bpp"],   label="Val bpp")
plt.xlabel("Epoch"); plt.ylabel("bpp")
plt.title("Bitrate Curve"); plt.legend(); plt.grid()
plt.savefig(f"{save_dir}/bpp_curve.png"); plt.show()

# ===== 率失真散点（每 epoch 一个点）=====
plt.figure()
plt.scatter(history["val_bpp"], history["val_psnr"], c=list(epochs_ran), cmap="viridis")
plt.colorbar(label="Epoch")
plt.xlabel("bpp"); plt.ylabel("PSNR (dB)")
plt.title("Rate-Distortion (Val)"); plt.grid()
plt.savefig(f"{save_dir}/rd_curve.png"); plt.show()

print(f"\n训练完成。最优 Val PSNR: {best_psnr:.2f} dB", flush=True)

In [ ]:
"""
任务联合压缩训练脚本
在 Cheng2020Anchor 率失真损失基础上，加入 YOLOv8 检测特征损失
损失函数：L = bpp + λ × 255² × MSE + μ × detection_loss

相较于 train_compressai.py 的改动：
  1. DFireDataset.__getitem__ 增加 bbox 读取
  2. collate_fn 处理变长 bbox
  3. 加载冻结的 YOLOv8n 作为感知损失计算器
  4. 训练循环加入 detection_loss 项
  5. history / 打印 / 绘图增加 det_loss 追踪
  其余（优化器、scheduler、早停、绘图）原样保留
"""
!pip install ultralytics
import glob
import os
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
import torch.optim as optim
from PIL import Image
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import transforms

from compressai.models import Cheng2020Anchor

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(iterable, **kwargs):
        return iterable

# ===== 依赖检查 =====
try:
    from ultralytics import YOLO
except ImportError:
    raise ImportError("请先运行: pip install ultralytics")

# =====================================================================
# 配置
# =====================================================================
IMG_SIZE          = 384       # Cheng2020 要求 64 的倍数
BATCH_SIZE        = 8         # 加了 YOLO 后显存增加，从 16 降到 8
NUM_WORKERS       = 0
MAX_TRAIN_SAMPLES = 2048
MAX_VAL_SAMPLES   = 512
LOG_EVERY         = 20
DATA_ROOT         = "/content/D-Fire"
EPOCHS            = 30
EARLY_STOP_PATIENCE = 5

LAMBDA  = 0.05    # 率失真权衡，沿用上一次最优配置
MU      = 0.1     # 检测任务损失权重，从 0.1 开始；若 PSNR 下降过多可调小到 0.05

LR_MAIN = 5e-5    # 沿用上一次最优配置
LR_AUX  = 1e-3
GRAD_CLIP_NORM = 1.0

save_dir = "outputs_task_aware"
os.makedirs(save_dir, exist_ok=True)

history = {
    "train_loss": [], "val_loss": [],
    "train_psnr": [], "val_psnr": [],
    "train_bpp":  [], "val_bpp":  [],
    "train_det":  [], "val_det":  [],   # 新增：检测损失追踪
}

# =====================================================================
# 设备
# =====================================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    torch.backends.cudnn.benchmark = True
print(f"Device: {device}", flush=True)
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}", flush=True)

# =====================================================================
# 数据集
# 改动：__getitem__ 增加 bbox 读取（YOLO 格式 txt）
# =====================================================================
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.Lambda(lambda img: img.convert("RGB") if img.mode != "RGB" else img),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.Lambda(lambda img: img.convert("RGB") if img.mode != "RGB" else img),
    transforms.ToTensor(),
])


class DFireDataset(Dataset):
    """
    D-Fire: root/{train,test}/images/*.jpg
           root/{train,test}/labels/*.txt   ← YOLO 格式：class cx cy w h（归一化）
    若某张图没有对应 label 文件（负样本），boxes 返回空 tensor。
    """
    def __init__(self, root, split="train", transform=None):
        assert split in ("train", "test")
        img_dir = os.path.join(root, split, "images")
        self.lbl_dir = os.path.join(root, split, "labels")
        self.paths = sorted(
            glob.glob(os.path.join(img_dir, "*.jpg"))
            + glob.glob(os.path.join(img_dir, "*.png"))
        )
        if not self.paths:
            raise FileNotFoundError(f"在 {img_dir} 下找不到图片")
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img_path = self.paths[idx]
        img = Image.open(img_path).convert("RGB")
        if self.transform:
            img = self.transform(img)

        # 读取对应 label（可能不存在）
        stem = os.path.splitext(os.path.basename(img_path))[0]
        lbl_path = os.path.join(self.lbl_dir, stem + ".txt")
        boxes = []
        if os.path.exists(lbl_path):
            with open(lbl_path) as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) == 5:
                        # YOLO格式：class cx cy w h，全部归一化到 [0,1]
                        boxes.append([float(p) for p in parts])
        boxes = torch.tensor(boxes, dtype=torch.float32)  # [N, 5] 或 [0, 5]
        return img, boxes


def collate_fn(batch):
    """
    DataLoader 默认 collate 无法处理变长 bbox，需要自定义。
    imgs:  [B, 3, H, W]
    boxes: list of [N_i, 5]（每张图的 bbox 数量不同）
    """
    imgs, boxes = zip(*batch)
    imgs = torch.stack(imgs, 0)
    return imgs, list(boxes)


def maybe_subset(dataset, max_samples):
    if max_samples is None or max_samples >= len(dataset):
        return dataset
    return Subset(dataset, list(range(max_samples)))


@torch.no_grad()
def batch_psnr(recon, target):
    mse = (recon - target).pow(2).mean(dim=(1, 2, 3))
    return (10 * torch.log10(1.0 / mse.clamp(min=1e-8))).mean().item()


print("Loading dataset...", flush=True)
train_db = maybe_subset(
    DFireDataset(DATA_ROOT, split="train", transform=train_transform),
    MAX_TRAIN_SAMPLES,
)
val_db = maybe_subset(
    DFireDataset(DATA_ROOT, split="test", transform=val_transform),
    MAX_VAL_SAMPLES,
)

pin_memory = device.type == "cuda"
train_loader = DataLoader(train_db, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=pin_memory,
                          collate_fn=collate_fn)
val_loader   = DataLoader(val_db,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=pin_memory,
                          collate_fn=collate_fn)

print(
    f"Train: {len(train_db)} imgs | Val: {len(val_db)} imgs | "
    f"{IMG_SIZE}x{IMG_SIZE} bs={BATCH_SIZE}",
    flush=True,
)

# =====================================================================
# 压缩模型
# =====================================================================
print("Building compression model...", flush=True)
model = Cheng2020Anchor(N=128).to(device)

# 加载上一步训练好的最优权重，在此基础上微调
PRETRAINED_CKPT = "outputs_compressai/best_model.pth"
if os.path.exists(PRETRAINED_CKPT):
    ckpt = torch.load(PRETRAINED_CKPT, map_location=device)
    model.load_state_dict(ckpt["model"])
    print(f"已加载预训练权重：{PRETRAINED_CKPT}（来自 epoch {ckpt['epoch']}，"
          f"PSNR={ckpt['val_psnr']:.2f} dB）", flush=True)
else:
    print(f"未找到预训练权重 {PRETRAINED_CKPT}，从头训练。", flush=True)

# =====================================================================
# YOLOv8 感知损失计算器
# 冻结权重，只用中间特征来计算检测损失信号
# =====================================================================
print("Loading YOLOv8n feature extractor...", flush=True)
yolo = YOLO("yolov8n.pt")          # 首次运行会自动下载，约 6 MB
yolo_model = yolo.model.to(device)
yolo_model.eval()
for p in yolo_model.parameters():  # 完全冻结，不参与梯度更新
    p.requires_grad = False

def detection_feature_loss(x_hat, x_orig):
    feats_hat  = {}
    feats_orig = {}
    target_layers = [2, 4]

    def make_hook(store, key):
        def hook(module, inp, out):
            store[key] = out
        return hook

    # 注册 hook
    hooks = []
    for i, layer in enumerate(yolo_model.model):
        if i in target_layers:
            hooks.append(layer.register_forward_hook(make_hook(feats_hat, i)))

    # 跑重建图（需要梯度，不加 no_grad）
    _ = yolo_model(x_hat)
    for h in hooks:
        h.remove()

    # 注册 hook 跑原图（不需要梯度）
    hooks = []
    for i, layer in enumerate(yolo_model.model):
        if i in target_layers:
            hooks.append(layer.register_forward_hook(make_hook(feats_orig, i)))

    with torch.no_grad():
        _ = yolo_model(x_orig)
    for h in hooks:
        h.remove()

    # 计算特征距离
    feat_loss = torch.tensor(0.0, device=x_hat.device)
    for k in target_layers:
        feat_loss = feat_loss + F.mse_loss(feats_hat[k], feats_orig[k].detach())

    return feat_loss / len(target_layers)



# =====================================================================
# 优化器（原样保留双优化器结构）
# =====================================================================
optimizer     = optim.Adam(model.parameters(), lr=LR_MAIN)
aux_optimizer = optim.Adam(model.entropy_bottleneck.parameters(), lr=LR_AUX)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=2, min_lr=1e-7
)

# =====================================================================
# 训练循环
# =====================================================================
best_psnr         = 0.0
epochs_no_improve = 0

for epoch in range(EPOCHS):
    t_epoch = time.time()

    # ---------- Train ----------
    model.train()
    total_loss = total_bpp = total_psnr = total_det = 0.0
    n_batches = 0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [train]", leave=False)
    for step, (imgs, boxes) in enumerate(pbar, 1):
        imgs = imgs.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        aux_optimizer.zero_grad(set_to_none=True)

        out   = model(imgs)
        x_hat = out["x_hat"].clamp(0, 1)

        num_pixels = imgs.shape[0] * imgs.shape[2] * imgs.shape[3]
        bpp = sum(
            (-torch.log2(lk).sum() / num_pixels)
            for lk in out["likelihoods"].values()
        )

        distortion  = F.mse_loss(x_hat, imgs)
        det_loss    = detection_feature_loss(x_hat, imgs)

        # 联合损失：率失真 + 检测感知
        loss = bpp + LAMBDA * 255**2 * distortion + MU * det_loss

        loss.backward()
        if GRAD_CLIP_NORM:
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
        optimizer.step()

        aux_loss = model.entropy_bottleneck.loss()
        aux_loss.backward()
        aux_optimizer.step()

        total_loss += loss.item()
        total_bpp  += bpp.item()
        total_det  += det_loss.item()
        with torch.no_grad():
            total_psnr += batch_psnr(x_hat, imgs)
        n_batches += 1

        if step % LOG_EVERY == 0:
            pbar.set_postfix(
                loss=f"{loss.item():.4f}",
                bpp=f"{bpp.item():.3f}",
                det=f"{det_loss.item():.4f}",
            )

    avg_train_loss = total_loss / n_batches
    avg_train_bpp  = total_bpp  / n_batches
    avg_train_psnr = total_psnr / n_batches
    avg_train_det  = total_det  / n_batches

    # ---------- Validation ----------
    model.eval()
    val_loss = val_bpp = val_psnr = val_det = 0.0
    n_val = 0

    with torch.no_grad():
        for imgs, boxes in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [val]", leave=False):
            imgs  = imgs.to(device, non_blocking=True)
            out   = model(imgs)
            x_hat = out["x_hat"].clamp(0, 1)

            num_pixels = imgs.shape[0] * imgs.shape[2] * imgs.shape[3]
            bpp = sum(
                (-torch.log2(lk).sum() / num_pixels)
                for lk in out["likelihoods"].values()
            )
            distortion = F.mse_loss(x_hat, imgs)
            det_loss   = detection_feature_loss(x_hat, imgs)
            loss       = bpp + LAMBDA * 255**2 * distortion + MU * det_loss

            val_loss += loss.item()
            val_bpp  += bpp.item()
            val_psnr += batch_psnr(x_hat, imgs)
            val_det  += det_loss.item()
            n_val    += 1

    avg_val_loss = val_loss / n_val
    avg_val_bpp  = val_bpp  / n_val
    avg_val_psnr = val_psnr / n_val
    avg_val_det  = val_det  / n_val
    elapsed = time.time() - t_epoch

    for k, v in zip(
        ["train_loss","val_loss","train_psnr","val_psnr",
         "train_bpp","val_bpp","train_det","val_det"],
        [avg_train_loss, avg_val_loss, avg_train_psnr, avg_val_psnr,
         avg_train_bpp,  avg_val_bpp,  avg_train_det,  avg_val_det],
    ):
        history[k].append(v)

    scheduler.step(avg_val_psnr)

    print(
        f"Epoch {epoch+1}/{EPOCHS} ({elapsed:.1f}s) | "
        f"Train Loss {avg_train_loss:.4f} | Val Loss {avg_val_loss:.4f} | "
        f"Train PSNR {avg_train_psnr:.2f} | Val PSNR {avg_val_psnr:.2f} dB | "
        f"Val bpp {avg_val_bpp:.4f} | Val det {avg_val_det:.4f}",
        flush=True,
    )

    if avg_val_psnr > best_psnr:
        best_psnr = avg_val_psnr
        epochs_no_improve = 0
        torch.save(
            {
                "epoch": epoch + 1,
                "model": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "val_psnr": avg_val_psnr,
                "val_bpp":  avg_val_bpp,
                "val_det":  avg_val_det,
            },
            f"{save_dir}/best_model.pth",
        )
        print(f"  → 保存最优模型 PSNR={best_psnr:.2f} dB  "
              f"bpp={avg_val_bpp:.4f}  det={avg_val_det:.4f}", flush=True)
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= EARLY_STOP_PATIENCE:
            print(
                f"Early stop: Val PSNR 连续 {EARLY_STOP_PATIENCE} epoch "
                f"未超过 {best_psnr:.2f} dB",
                flush=True,
            )
            break

# =====================================================================
# 绘图（在原有三张图基础上增加检测损失曲线）
# =====================================================================
epochs_ran = range(1, len(history["train_loss"]) + 1)

plt.figure()
plt.plot(epochs_ran, history["train_loss"], label="Train Loss")
plt.plot(epochs_ran, history["val_loss"],   label="Val Loss")
plt.xlabel("Epoch"); plt.ylabel("Loss")
plt.title("Loss Curve"); plt.legend(); plt.grid()
plt.savefig(f"{save_dir}/loss_curve.png"); plt.show()

plt.figure()
plt.plot(epochs_ran, history["train_psnr"], label="Train PSNR")
plt.plot(epochs_ran, history["val_psnr"],   label="Val PSNR")
plt.xlabel("Epoch"); plt.ylabel("PSNR (dB)")
plt.title("PSNR Curve"); plt.legend(); plt.grid()
plt.savefig(f"{save_dir}/psnr_curve.png"); plt.show()

plt.figure()
plt.plot(epochs_ran, history["train_bpp"], label="Train bpp")
plt.plot(epochs_ran, history["val_bpp"],   label="Val bpp")
plt.xlabel("Epoch"); plt.ylabel("bpp")
plt.title("Bitrate Curve"); plt.legend(); plt.grid()
plt.savefig(f"{save_dir}/bpp_curve.png"); plt.show()

# 新增：检测感知损失曲线
plt.figure()
plt.plot(epochs_ran, history["train_det"], label="Train Det Loss")
plt.plot(epochs_ran, history["val_det"],   label="Val Det Loss")
plt.xlabel("Epoch"); plt.ylabel("Detection Feature Loss")
plt.title("Detection Loss Curve"); plt.legend(); plt.grid()
plt.savefig(f"{save_dir}/det_loss_curve.png"); plt.show()

plt.figure()
plt.scatter(history["val_bpp"], history["val_psnr"],
            c=list(epochs_ran), cmap="viridis")
plt.colorbar(label="Epoch")
plt.xlabel("bpp"); plt.ylabel("PSNR (dB)")
plt.title("Rate-Distortion (Val)"); plt.grid()
plt.savefig(f"{save_dir}/rd_curve.png"); plt.show()

print(f"\n训练完成。最优 Val PSNR: {best_psnr:.2f} dB", flush=True)

In [ ]:
"""
低 bpp 微调脚本 —— RDDM 实验第一步
在已训练好的 task-aware Cheng2020 权重基础上，调小 λ 继续微调，
把率失真平衡点推到 bpp≈0.15~0.2 的极低码率区间。

相较于 train_task_aware.py 的改动：
  1. LAMBDA 调小（0.05 → 0.008），目标 bpp 落在 0.15~0.2
  2. 加载 outputs_task_aware/best_model.pth 作为起点（而非从头训练）
  3. 修复了 detection_feature_loss 的 hook bug
     （原版原图/重建图特征被混在同一个 list 里，导致 det_loss 恒为 0）
  4. EPOCHS 调小（微调不需要 30 epoch 那么久）
  其余（数据集、优化器结构、绘图）原样保留
"""

import glob
import os
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
import torch.optim as optim
from PIL import Image
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import transforms

from compressai.models import Cheng2020Anchor

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(iterable, **kwargs):
        return iterable

# ===== 依赖检查 =====
try:
    from ultralytics import YOLO
except ImportError:
    raise ImportError("请先运行: pip install ultralytics")

# =====================================================================
# 配置
# =====================================================================
IMG_SIZE          = 384       # Cheng2020 要求 64 的倍数
BATCH_SIZE        = 8         # 加了 YOLO 后显存增加，从 16 降到 8
NUM_WORKERS       = 0
MAX_TRAIN_SAMPLES = 2048
MAX_VAL_SAMPLES   = 512
LOG_EVERY         = 20
DATA_ROOT         = "/content/D-Fire"
EPOCHS            = 15   # 微调不需要从头训练那么久
EARLY_STOP_PATIENCE = 5

# 目标：把 bpp 从 0.42 推到 0.15~0.2。
# 参考你之前的曲线：λ=0.05→bpp 0.42，λ=0.01→bpp~0.3，
# 所以这次需要比 0.01 更小，从 0.008 起步，如果第一轮 bpp 仍 >0.2 可再调小到 0.004。
LAMBDA  = 0.008
MU      = 0.1     # 检测损失权重不变，继续保留检测感知能力

LR_MAIN = 2e-5    # 微调用更小的学习率，避免把已学到的特征破坏掉
LR_AUX  = 5e-4
GRAD_CLIP_NORM = 1.0

save_dir = "outputs_low_bpp"
os.makedirs(save_dir, exist_ok=True)

history = {
    "train_loss": [], "val_loss": [],
    "train_psnr": [], "val_psnr": [],
    "train_bpp":  [], "val_bpp":  [],
    "train_det":  [], "val_det":  [],   # 新增：检测损失追踪
}

# =====================================================================
# 设备
# =====================================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    torch.backends.cudnn.benchmark = True
print(f"Device: {device}", flush=True)
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}", flush=True)

# =====================================================================
# 数据集
# 改动：__getitem__ 增加 bbox 读取（YOLO 格式 txt）
# =====================================================================
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.Lambda(lambda img: img.convert("RGB") if img.mode != "RGB" else img),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.Lambda(lambda img: img.convert("RGB") if img.mode != "RGB" else img),
    transforms.ToTensor(),
])


class DFireDataset(Dataset):
    """
    D-Fire: root/{train,test}/images/*.jpg
           root/{train,test}/labels/*.txt   ← YOLO 格式：class cx cy w h（归一化）
    若某张图没有对应 label 文件（负样本），boxes 返回空 tensor。
    """
    def __init__(self, root, split="train", transform=None):
        assert split in ("train", "test")
        img_dir = os.path.join(root, split, "images")
        self.lbl_dir = os.path.join(root, split, "labels")
        self.paths = sorted(
            glob.glob(os.path.join(img_dir, "*.jpg"))
            + glob.glob(os.path.join(img_dir, "*.png"))
        )
        if not self.paths:
            raise FileNotFoundError(f"在 {img_dir} 下找不到图片")
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img_path = self.paths[idx]
        img = Image.open(img_path).convert("RGB")
        if self.transform:
            img = self.transform(img)

        # 读取对应 label（可能不存在）
        stem = os.path.splitext(os.path.basename(img_path))[0]
        lbl_path = os.path.join(self.lbl_dir, stem + ".txt")
        boxes = []
        if os.path.exists(lbl_path):
            with open(lbl_path) as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) == 5:
                        # YOLO格式：class cx cy w h，全部归一化到 [0,1]
                        boxes.append([float(p) for p in parts])
        boxes = torch.tensor(boxes, dtype=torch.float32)  # [N, 5] 或 [0, 5]
        return img, boxes


def collate_fn(batch):
    """
    DataLoader 默认 collate 无法处理变长 bbox，需要自定义。
    imgs:  [B, 3, H, W]
    boxes: list of [N_i, 5]（每张图的 bbox 数量不同）
    """
    imgs, boxes = zip(*batch)
    imgs = torch.stack(imgs, 0)
    return imgs, list(boxes)


def maybe_subset(dataset, max_samples):
    if max_samples is None or max_samples >= len(dataset):
        return dataset
    return Subset(dataset, list(range(max_samples)))


@torch.no_grad()
def batch_psnr(recon, target):
    mse = (recon - target).pow(2).mean(dim=(1, 2, 3))
    return (10 * torch.log10(1.0 / mse.clamp(min=1e-8))).mean().item()


print("Loading dataset...", flush=True)
train_db = maybe_subset(
    DFireDataset(DATA_ROOT, split="train", transform=train_transform),
    MAX_TRAIN_SAMPLES,
)
val_db = maybe_subset(
    DFireDataset(DATA_ROOT, split="test", transform=val_transform),
    MAX_VAL_SAMPLES,
)

pin_memory = device.type == "cuda"
train_loader = DataLoader(train_db, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=pin_memory,
                          collate_fn=collate_fn)
val_loader   = DataLoader(val_db,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=pin_memory,
                          collate_fn=collate_fn)

print(
    f"Train: {len(train_db)} imgs | Val: {len(val_db)} imgs | "
    f"{IMG_SIZE}x{IMG_SIZE} bs={BATCH_SIZE}",
    flush=True,
)

# =====================================================================
# 压缩模型
# =====================================================================
print("Building compression model...", flush=True)
model = Cheng2020Anchor(N=128).to(device)

# ⚠️ 重点：必须加载 task-aware 权重（带检测损失训练出来的），
# 不是纯压缩基线 outputs_compressai/best_model.pth。
# 如果你按之前的建议把权重存到了 Drive，先挂载 Drive 再改这里的路径，例如：
#   from google.colab import drive; drive.mount('/content/drive')
#   PRETRAINED_CKPT = "/content/drive/MyDrive/dfire_checkpoints/cheng2020_task_aware_best.pth"
PRETRAINED_CKPT = "outputs_task_aware/best_model.pth"
if os.path.exists(PRETRAINED_CKPT):
    ckpt = torch.load(PRETRAINED_CKPT, map_location=device)
    model.load_state_dict(ckpt["model"])
    print(f"已加载 task-aware 权重：{PRETRAINED_CKPT}（来自 epoch {ckpt['epoch']}，"
          f"PSNR={ckpt['val_psnr']:.2f} dB, bpp={ckpt.get('val_bpp', float('nan')):.4f}）",
          flush=True)
else:
    raise FileNotFoundError(
        f"找不到 {PRETRAINED_CKPT}。这一步要求必须基于 task-aware 权重微调，"
        f"请确认路径，或从 Drive 拷回 /content 后再运行。"
    )

# =====================================================================
# YOLOv8 感知损失计算器
# 冻结权重，只用中间特征来计算检测损失信号
# =====================================================================
print("Loading YOLOv8n feature extractor...", flush=True)
yolo = YOLO("yolov8n.pt")          # 首次运行会自动下载，约 6 MB
yolo_model = yolo.model.to(device)
yolo_model.eval()
for p in yolo_model.parameters():  # 完全冻结，不参与梯度更新
    p.requires_grad = False


def detection_feature_loss(x_hat, x_orig):
    """
    用 YOLOv8 backbone 提取重建图和原图的中间特征，
    计算 L2 特征距离作为检测感知损失。

    [修复] 原版用两个 list 分别 append，但两次 forward 共用同一组 hook，
    导致 feats_hat 实际存的是两次 forward 的混合结果，距离恒为 0。
    这里改成按层索引存入字典，并将两次 forward 的 hook 完全分开注册/移除。

    只取第 2、4 层（浅层特征）：对应边缘/纹理等低级特征，
    正好是火焰/烟雾检测所需要的，计算量也更小。
    """
    target_layers = [2, 4]

    def make_hook(store, key):
        def hook(module, inp, out):
            store[key] = out
        return hook

    # ---- 第一次 forward：重建图，需要保留梯度 ----
    feats_hat = {}
    hooks = []
    for i, layer in enumerate(yolo_model.model):
        if i in target_layers:
            hooks.append(layer.register_forward_hook(make_hook(feats_hat, i)))
    _ = yolo_model(x_hat)
    for h in hooks:
        h.remove()

    # ---- 第二次 forward：原图，仅作参考，不需要梯度 ----
    feats_orig = {}
    hooks = []
    for i, layer in enumerate(yolo_model.model):
        if i in target_layers:
            hooks.append(layer.register_forward_hook(make_hook(feats_orig, i)))
    with torch.no_grad():
        _ = yolo_model(x_orig)
    for h in hooks:
        h.remove()

    feat_loss = torch.tensor(0.0, device=x_hat.device)
    for k in target_layers:
        feat_loss = feat_loss + F.mse_loss(feats_hat[k], feats_orig[k].detach())

    return feat_loss / len(target_layers)


# =====================================================================
# 优化器（原样保留双优化器结构）
# =====================================================================
optimizer     = optim.Adam(model.parameters(), lr=LR_MAIN)
aux_optimizer = optim.Adam(model.entropy_bottleneck.parameters(), lr=LR_AUX)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=2, min_lr=1e-7
)

# =====================================================================
# 训练循环
# =====================================================================
best_psnr         = 0.0
epochs_no_improve = 0

for epoch in range(EPOCHS):
    t_epoch = time.time()

    # ---------- Train ----------
    model.train()
    total_loss = total_bpp = total_psnr = total_det = 0.0
    n_batches = 0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [train]", leave=False)
    for step, (imgs, boxes) in enumerate(pbar, 1):
        imgs = imgs.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        aux_optimizer.zero_grad(set_to_none=True)

        out   = model(imgs)
        x_hat = out["x_hat"].clamp(0, 1)

        num_pixels = imgs.shape[0] * imgs.shape[2] * imgs.shape[3]
        bpp = sum(
            (-torch.log2(lk).sum() / num_pixels)
            for lk in out["likelihoods"].values()
        )

        distortion  = F.mse_loss(x_hat, imgs)
        det_loss    = detection_feature_loss(x_hat, imgs)

        # 联合损失：率失真 + 检测感知
        loss = bpp + LAMBDA * 255**2 * distortion + MU * det_loss

        loss.backward()
        if GRAD_CLIP_NORM:
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
        optimizer.step()

        aux_loss = model.entropy_bottleneck.loss()
        aux_loss.backward()
        aux_optimizer.step()

        total_loss += loss.item()
        total_bpp  += bpp.item()
        total_det  += det_loss.item()
        with torch.no_grad():
            total_psnr += batch_psnr(x_hat, imgs)
        n_batches += 1

        if step % LOG_EVERY == 0:
            pbar.set_postfix(
                loss=f"{loss.item():.4f}",
                bpp=f"{bpp.item():.3f}",
                det=f"{det_loss.item():.4f}",
            )

    avg_train_loss = total_loss / n_batches
    avg_train_bpp  = total_bpp  / n_batches
    avg_train_psnr = total_psnr / n_batches
    avg_train_det  = total_det  / n_batches

    # ---------- Validation ----------
    model.eval()
    val_loss = val_bpp = val_psnr = val_det = 0.0
    n_val = 0

    with torch.no_grad():
        for imgs, boxes in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [val]", leave=False):
            imgs  = imgs.to(device, non_blocking=True)
            out   = model(imgs)
            x_hat = out["x_hat"].clamp(0, 1)

            num_pixels = imgs.shape[0] * imgs.shape[2] * imgs.shape[3]
            bpp = sum(
                (-torch.log2(lk).sum() / num_pixels)
                for lk in out["likelihoods"].values()
            )
            distortion = F.mse_loss(x_hat, imgs)
            det_loss   = detection_feature_loss(x_hat, imgs)
            loss       = bpp + LAMBDA * 255**2 * distortion + MU * det_loss

            val_loss += loss.item()
            val_bpp  += bpp.item()
            val_psnr += batch_psnr(x_hat, imgs)
            val_det  += det_loss.item()
            n_val    += 1

    avg_val_loss = val_loss / n_val
    avg_val_bpp  = val_bpp  / n_val
    avg_val_psnr = val_psnr / n_val
    avg_val_det  = val_det  / n_val
    elapsed = time.time() - t_epoch

    for k, v in zip(
        ["train_loss","val_loss","train_psnr","val_psnr",
         "train_bpp","val_bpp","train_det","val_det"],
        [avg_train_loss, avg_val_loss, avg_train_psnr, avg_val_psnr,
         avg_train_bpp,  avg_val_bpp,  avg_train_det,  avg_val_det],
    ):
        history[k].append(v)

    scheduler.step(avg_val_psnr)

    print(
        f"Epoch {epoch+1}/{EPOCHS} ({elapsed:.1f}s) | "
        f"Train Loss {avg_train_loss:.4f} | Val Loss {avg_val_loss:.4f} | "
        f"Train PSNR {avg_train_psnr:.2f} | Val PSNR {avg_val_psnr:.2f} dB | "
        f"Val bpp {avg_val_bpp:.4f} | Val det {avg_val_det:.4f}",
        flush=True,
    )

    if avg_val_psnr > best_psnr:
        best_psnr = avg_val_psnr
        epochs_no_improve = 0
        torch.save(
            {
                "epoch": epoch + 1,
                "model": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "val_psnr": avg_val_psnr,
                "val_bpp":  avg_val_bpp,
                "val_det":  avg_val_det,
            },
            f"{save_dir}/best_model.pth",
        )
        print(f"  → 保存最优模型 PSNR={best_psnr:.2f} dB  "
              f"bpp={avg_val_bpp:.4f}  det={avg_val_det:.4f}", flush=True)
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= EARLY_STOP_PATIENCE:
            print(
                f"Early stop: Val PSNR 连续 {EARLY_STOP_PATIENCE} epoch "
                f"未超过 {best_psnr:.2f} dB",
                flush=True,
            )
            break

# =====================================================================
# 绘图（在原有三张图基础上增加检测损失曲线）
# =====================================================================
epochs_ran = range(1, len(history["train_loss"]) + 1)

plt.figure()
plt.plot(epochs_ran, history["train_loss"], label="Train Loss")
plt.plot(epochs_ran, history["val_loss"],   label="Val Loss")
plt.xlabel("Epoch"); plt.ylabel("Loss")
plt.title("Loss Curve"); plt.legend(); plt.grid()
plt.savefig(f"{save_dir}/loss_curve.png"); plt.show()

plt.figure()
plt.plot(epochs_ran, history["train_psnr"], label="Train PSNR")
plt.plot(epochs_ran, history["val_psnr"],   label="Val PSNR")
plt.xlabel("Epoch"); plt.ylabel("PSNR (dB)")
plt.title("PSNR Curve"); plt.legend(); plt.grid()
plt.savefig(f"{save_dir}/psnr_curve.png"); plt.show()

plt.figure()
plt.plot(epochs_ran, history["train_bpp"], label="Train bpp")
plt.plot(epochs_ran, history["val_bpp"],   label="Val bpp")
plt.xlabel("Epoch"); plt.ylabel("bpp")
plt.title("Bitrate Curve"); plt.legend(); plt.grid()
plt.savefig(f"{save_dir}/bpp_curve.png"); plt.show()

# 新增：检测感知损失曲线
plt.figure()
plt.plot(epochs_ran, history["train_det"], label="Train Det Loss")
plt.plot(epochs_ran, history["val_det"],   label="Val Det Loss")
plt.xlabel("Epoch"); plt.ylabel("Detection Feature Loss")
plt.title("Detection Loss Curve"); plt.legend(); plt.grid()
plt.savefig(f"{save_dir}/det_loss_curve.png"); plt.show()

plt.figure()
plt.scatter(history["val_bpp"], history["val_psnr"],
            c=list(epochs_ran), cmap="viridis")
plt.colorbar(label="Epoch")
plt.xlabel("bpp"); plt.ylabel("PSNR (dB)")
plt.title("Rate-Distortion (Val)"); plt.grid()
plt.savefig(f"{save_dir}/rd_curve.png"); plt.show()

print(f"\n训练完成。最优 Val PSNR: {best_psnr:.2f} dB", flush=True)

In [ ]:
"""
第一步：在 D-Fire 上快速微调 YOLOv8n 检测器
目的：得到一个"能用就行"的检测器，用于后续对比原图/压缩重建图的检测精度差异。
不追求绝对 mAP 数值，只追求三种图像之间误差趋势的相对可比性。

D-Fire 标注格式确认：YOLO txt，class 0=Fire, 1=Smoke
目录结构：/content/D-Fire/{train,test}/{images,labels}
"""
!pip install "numpy>=1.23.0,<2.0"
import os
import yaml
from ultralytics import YOLO

DATA_ROOT = "/content/D-Fire"
EPOCHS    = 8          # 快速微调，重点是"能用"而非"最优"
IMG_SIZE  = 384         # 和压缩模型训练时保持一致，三方对比口径统一
BATCH     = 16

# ---------------------------------------------------------------
# D-Fire 官方约定：0=Fire, 1=Smoke
# 如果训练效果异常（比如所有框被识别成同一类），
# 优先检查这里的映射是否与实际标注一致。
# ---------------------------------------------------------------
data_yaml = {
    "path":  DATA_ROOT,
    "train": "train/images",
    "val":   "test/images",     # D-Fire 只有 train/test，没有单独 val，这里复用 test
    "names": {0: "fire", 1: "smoke"},
}

yaml_path = os.path.join(DATA_ROOT, "dfire.yaml")
with open(yaml_path, "w") as f:
    yaml.dump(data_yaml, f)
print(f"已生成 {yaml_path}")
print(yaml.dump(data_yaml))

# ---------------------------------------------------------------
# 微调（从 COCO 预训练权重开始，而不是随机初始化）
# ---------------------------------------------------------------
model = YOLO("yolov8n.pt")

results = model.train(
    data=yaml_path,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    project="dfire_detector",
    name="yolov8n_dfire_quick",
    patience=3,         # 快速实验，早停更激进一点
    save=True,
    plots=True,
    verbose=True,
)

print("\n训练完成，最优权重路径：")
print("dfire_detector/yolov8n_dfire_quick/weights/best.pt")

# ---------------------------------------------------------------
# 快速验证一下检测器本身的 mAP（在原图上），
# 这个数字作为后续对比实验的"地板参照"
# ---------------------------------------------------------------
# 改成
best_model = YOLO("/content/runs/detect/dfire_detector/yolov8n_dfire_quick/weights/best.pt")
metrics = best_model.val(data=yaml_path, imgsz=IMG_SIZE)

print(f"\n检测器在原图(D-Fire test集)上的表现：")
print(f"  mAP50    : {metrics.box.map50:.4f}")
print(f"  mAP50-95 : {metrics.box.map:.4f}")

In [ ]:
"""
可视化诊断脚本
从 D-Fire test 集里按类别挑几张代表性图片，
并排显示 原图 / task-aware重建图(bpp=0.42) / low-bpp重建图(bpp=0.18)
用于肉眼判断：mAP 大幅下降是"细节模糊"还是"语义结构丢失"。

运行前提：
  - outputs_task_aware / outputs_low_bpp 权重已加载（沿用之前评估脚本的变量），
    如果是新会话，先重新跑权重加载部分。
  - D-Fire 标注在 train/test 的 labels 目录下（YOLO格式：class cx cy w h）
"""

import os
import glob
import random

import torch
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms

from compressai.models import Cheng2020Anchor

# =====================================================================
# 配置
# =====================================================================
DATA_ROOT  = "/content/D-Fire"
IMG_SIZE   = 384
DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")

TASK_AWARE_CKPT = "/content/cheng2020_task_aware_best.pth"
LOW_BPP_CKPT    = "/content/cheng2020_low_bpp_best.pth"

N_PER_CATEGORY = 2   # 每类挑几张

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.Lambda(lambda img: img.convert("RGB") if img.mode != "RGB" else img),
    transforms.ToTensor(),
])


# =====================================================================
# 按类别分类图片：fire / smoke / both / none
# =====================================================================
def categorize_images(root, split="test"):
    img_dir = os.path.join(root, split, "images")
    lbl_dir = os.path.join(root, split, "labels")
    paths = sorted(
        glob.glob(os.path.join(img_dir, "*.jpg"))
        + glob.glob(os.path.join(img_dir, "*.png"))
    )

    categories = {"fire": [], "smoke": [], "both": [], "none": []}

    for p in paths:
        stem = os.path.splitext(os.path.basename(p))[0]
        lbl_path = os.path.join(lbl_dir, stem + ".txt")
        classes = set()
        if os.path.exists(lbl_path):
            with open(lbl_path) as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) == 5:
                        classes.add(int(parts[0]))

        if not classes:
            categories["none"].append(p)
        elif classes == {0}:
            categories["fire"].append(p)
        elif classes == {1}:
            categories["smoke"].append(p)
        else:
            categories["both"].append(p)

    return categories


print("正在扫描并分类图片...", flush=True)
categories = categorize_images(DATA_ROOT)
for k, v in categories.items():
    print(f"  {k}: {len(v)} 张")

# 随机挑选（固定种子，方便复现）
random.seed(42)
selected_paths = []
selected_labels = []
for cat, paths in categories.items():
    if not paths:
        continue
    chosen = random.sample(paths, min(N_PER_CATEGORY, len(paths)))
    selected_paths.extend(chosen)
    selected_labels.extend([cat] * len(chosen))

print(f"\n共挑选 {len(selected_paths)} 张图片用于可视化对比")


# =====================================================================
# 加载两个压缩模型
# =====================================================================
print("加载压缩模型...", flush=True)
model_ta = Cheng2020Anchor(N=128).to(DEVICE)
ckpt_ta = torch.load(TASK_AWARE_CKPT, map_location=DEVICE)
model_ta.load_state_dict(ckpt_ta["model"])
model_ta.eval()

model_lb = Cheng2020Anchor(N=128).to(DEVICE)
ckpt_lb = torch.load(LOW_BPP_CKPT, map_location=DEVICE)
model_lb.load_state_dict(ckpt_lb["model"])
model_lb.eval()


# =====================================================================
# 对每张图生成两个重建版本，并三联画图
# =====================================================================
@torch.no_grad()
def reconstruct(model, img_tensor):
    out = model(img_tensor.unsqueeze(0).to(DEVICE))
    return out["x_hat"].clamp(0, 1).squeeze(0).cpu()


def to_numpy_img(tensor):
    return tensor.permute(1, 2, 0).numpy()


n = len(selected_paths)
fig, axes = plt.subplots(n, 3, figsize=(12, 4 * n))
if n == 1:
    axes = axes.reshape(1, 3)

for row, (path, label) in enumerate(zip(selected_paths, selected_labels)):
    img = Image.open(path).convert("RGB")
    img_tensor = transform(img)

    recon_ta = reconstruct(model_ta, img_tensor)
    recon_lb = reconstruct(model_lb, img_tensor)

    stem = os.path.splitext(os.path.basename(path))[0]

    axes[row, 0].imshow(to_numpy_img(img_tensor))
    axes[row, 0].set_title(f"原图 [{label}]\n{stem}", fontsize=9)
    axes[row, 0].axis("off")

    axes[row, 1].imshow(to_numpy_img(recon_ta))
    axes[row, 1].set_title(f"task-aware (bpp=0.42)\nPSNR={ckpt_ta['val_psnr']:.1f}", fontsize=9)
    axes[row, 1].axis("off")

    axes[row, 2].imshow(to_numpy_img(recon_lb))
    axes[row, 2].set_title(f"low-bpp (bpp=0.18)\nPSNR={ckpt_lb['val_psnr']:.1f}", fontsize=9)
    axes[row, 2].axis("off")

plt.tight_layout()
out_path = "/content/visual_diagnosis.png"
plt.savefig(out_path, dpi=150, bbox_inches="tight")
plt.show()

print(f"\n已保存对比图到: {out_path}")
print("\n看图时重点关注：")
print("  1. 火焰边缘是否模糊/锯齿化（细节模糊，RDDM能解决）")
print("  2. 烟雾的羽状边界是否变成色块（细节模糊，RDDM能解决）")
print("  3. 是否有目标位置错位、目标消失、或凭空出现伪影（语义结构丢失，RDDM未必能解决）")
print("  4. 'none'类别图片是否被压缩引入了类似烟雾/火焰的伪影（误检风险信号）")

In [ ]:
"""
检测框可视化诊断脚本
在 visual_diagnosis.py 基础上，加上 YOLOv8 检测器的实际推理结果，
把预测框和置信度画在图上，三联对比：原图 / task-aware重建图 / low-bpp重建图

目的：区分两种情况——
  (a) 完全漏检（框消失）       → 检测器对压缩失真完全失效，问题更严重
  (b) 检测到但置信度降低/框变松 → 压缩只是削弱了信号，RDDM补细节的思路更有希望见效

运行前提：
  - 已经跑过 train_detector.py，得到检测器权重
  - 已经跑过两个压缩模型的训练，得到 task-aware / low-bpp 权重
"""

import os
import glob
import random

import torch
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from torchvision import transforms

from compressai.models import Cheng2020Anchor
from ultralytics import YOLO

# =====================================================================
# 配置
# =====================================================================
DATA_ROOT  = "/content/D-Fire"
IMG_SIZE   = 384
DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DETECTOR_CKPT   = "/content/runs/detect/dfire_detector/yolov8n_dfire_quick/weights/best.pt"
TASK_AWARE_CKPT = "/content/cheng2020_task_aware_best.pth"
LOW_BPP_CKPT    = "/content/cheng2020_low_bpp_best.pth"

N_PER_CATEGORY  = 2
CONF_THRESHOLD  = 0.25     # 和 YOLO val 默认一致，低于此置信度的框不显示
CLASS_NAMES     = {0: "fire", 1: "smoke"}
CLASS_COLORS    = {0: "red", 1: "gray"}

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.Lambda(lambda img: img.convert("RGB") if img.mode != "RGB" else img),
    transforms.ToTensor(),
])


# =====================================================================
# 按类别分类图片（与 visual_diagnosis.py 相同逻辑）
# =====================================================================
def categorize_images(root, split="test"):
    img_dir = os.path.join(root, split, "images")
    lbl_dir = os.path.join(root, split, "labels")
    paths = sorted(
        glob.glob(os.path.join(img_dir, "*.jpg"))
        + glob.glob(os.path.join(img_dir, "*.png"))
    )

    categories = {"fire": [], "smoke": [], "both": [], "none": []}
    for p in paths:
        stem = os.path.splitext(os.path.basename(p))[0]
        lbl_path = os.path.join(lbl_dir, stem + ".txt")
        classes = set()
        if os.path.exists(lbl_path):
            with open(lbl_path) as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) == 5:
                        classes.add(int(parts[0]))
        if not classes:
            categories["none"].append(p)
        elif classes == {0}:
            categories["fire"].append(p)
        elif classes == {1}:
            categories["smoke"].append(p)
        else:
            categories["both"].append(p)
    return categories


print("正在扫描并分类图片...", flush=True)
categories = categorize_images(DATA_ROOT)
for k, v in categories.items():
    print(f"  {k}: {len(v)} 张")

random.seed(42)   # 固定种子，和上次可视化挑同一批图，方便对照
selected_paths = []
selected_labels = []
for cat, paths in categories.items():
    if not paths:
        continue
    chosen = random.sample(paths, min(N_PER_CATEGORY, len(paths)))
    selected_paths.extend(chosen)
    selected_labels.extend([cat] * len(chosen))

print(f"\n共挑选 {len(selected_paths)} 张图片")


# =====================================================================
# 加载模型：检测器 + 两个压缩模型
# =====================================================================
print("加载检测器...", flush=True)
detector = YOLO(DETECTOR_CKPT)

print("加载压缩模型...", flush=True)
model_ta = Cheng2020Anchor(N=128).to(DEVICE)
ckpt_ta = torch.load(TASK_AWARE_CKPT, map_location=DEVICE)
model_ta.load_state_dict(ckpt_ta["model"])
model_ta.eval()

model_lb = Cheng2020Anchor(N=128).to(DEVICE)
ckpt_lb = torch.load(LOW_BPP_CKPT, map_location=DEVICE)
model_lb.load_state_dict(ckpt_lb["model"])
model_lb.eval()


@torch.no_grad()
def reconstruct(model, img_tensor):
    out = model(img_tensor.unsqueeze(0).to(DEVICE))
    return out["x_hat"].clamp(0, 1).squeeze(0).cpu()


def to_numpy_img(tensor):
    return tensor.permute(1, 2, 0).numpy()


def draw_detections(ax, img_np, img_tensor, title):
    """在给定 axes 上画图，并叠加检测器的预测框"""
    ax.imshow(img_np)
    ax.set_title(title, fontsize=9)
    ax.axis("off")

    # YOLO 推理：直接传 tensor 即可，内部会处理归一化等
    results = detector.predict(
        img_tensor.unsqueeze(0).to(DEVICE),
        conf=CONF_THRESHOLD,
        verbose=False,
    )[0]

    n_boxes = 0
    if results.boxes is not None and len(results.boxes) > 0:
        h, w = img_np.shape[:2]
        for box in results.boxes:
            cls_id = int(box.cls.item())
            conf   = float(box.conf.item())
            x1, y1, x2, y2 = box.xyxy[0].tolist()  # 已经是像素坐标（基于输入图尺寸）

            color = CLASS_COLORS.get(cls_id, "yellow")
            rect = patches.Rectangle(
                (x1, y1), x2 - x1, y2 - y1,
                linewidth=1.5, edgecolor=color, facecolor="none"
            )
            ax.add_patch(rect)
            ax.text(
                x1, max(y1 - 4, 0),
                f"{CLASS_NAMES.get(cls_id, cls_id)} {conf:.2f}",
                color=color, fontsize=7,
                bbox=dict(facecolor="black", alpha=0.5, pad=1, edgecolor="none"),
            )
            n_boxes += 1

    return n_boxes


# =====================================================================
# 主循环：每张图三联画图 + 打印检测框数量对比
# =====================================================================
n = len(selected_paths)
fig, axes = plt.subplots(n, 3, figsize=(13, 4.2 * n))
if n == 1:
    axes = axes.reshape(1, 3)

print("\n" + "=" * 70)
print(f"{'图片':<14}{'类别':<8}{'原图框数':>10}{'task-aware框数':>16}{'low-bpp框数':>14}")
print("=" * 70)

for row, (path, label) in enumerate(zip(selected_paths, selected_labels)):
    img = Image.open(path).convert("RGB")
    img_tensor = transform(img)

    recon_ta = reconstruct(model_ta, img_tensor)
    recon_lb = reconstruct(model_lb, img_tensor)

    stem = os.path.splitext(os.path.basename(path))[0]

    n_orig = draw_detections(
        axes[row, 0], to_numpy_img(img_tensor), img_tensor,
        f"原图 [{label}]\n{stem}"
    )
    n_ta = draw_detections(
        axes[row, 1], to_numpy_img(recon_ta), recon_ta,
        f"task-aware (bpp=0.42)"
    )
    n_lb = draw_detections(
        axes[row, 2], to_numpy_img(recon_lb), recon_lb,
        f"low-bpp (bpp=0.18)"
    )

    print(f"{stem:<14}{label:<8}{n_orig:>10}{n_ta:>16}{n_lb:>14}")

print("=" * 70)

plt.tight_layout()
out_path = "/content/detection_diagnosis.png"
plt.savefig(out_path, dpi=150, bbox_inches="tight")
plt.show()

print(f"\n已保存检测框对比图到: {out_path}")
print("\n判断参考：")
print("  框数量相同但置信度明显降低 → 信号被削弱但结构保留，RDDM思路有希望")
print("  框数量减少（漏检）        → 部分目标特征已被压缩破坏到检测器无法识别")
print("  框数量增加（误检）        → 压缩引入的伪影被误判为目标，需警惕")

In [ ]:
"""
RDDM 数据对生成脚本 —— 第二步

用 task-aware (bpp≈0.42) 和 low-bpp (bpp≈0.18) 两个压缩模型，
分别在同一批图片上做重建，保存 (原图, 重建图) 配对，作为 RDDM 的训练数据。

RDDM 学习目标是从重建图预测残差 r = 原图 - 重建图，
所以这里把两者都原样保存（不计算残差），残差留给 RDDM 训练脚本自己算，
这样数据对更通用，也方便后续检查重建质量本身。

输出目录结构：
  /content/rddm_data/
    task_aware/
      train/orig/xxx.png      train/recon/xxx.png
      val/orig/xxx.png        val/recon/xxx.png
    low_bpp/
      train/orig/xxx.png      train/recon/xxx.png
      val/orig/xxx.png        val/recon/xxx.png

样本范围：与之前训练压缩模型时一致——
  train: D-Fire train/images 前 2048 张
  val:   D-Fire test/images  前 512 张
"""

import os
import glob

import torch
from PIL import Image
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import transforms
from torchvision.utils import save_image

from compressai.models import Cheng2020Anchor

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(iterable, **kwargs):
        return iterable

# =====================================================================
# 配置（与之前压缩模型训练保持一致，方便结果可比）
# =====================================================================
DATA_ROOT  = "/content/D-Fire"
IMG_SIZE   = 384
BATCH_SIZE = 16
NUM_WORKERS = 0
DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MAX_TRAIN_SAMPLES = 2048
MAX_VAL_SAMPLES   = 512

TASK_AWARE_CKPT = "/content/cheng2020_task_aware_best.pth"
LOW_BPP_CKPT    = "/content/cheng2020_low_bpp_best.pth"

OUT_ROOT = "/content/rddm_data"

# 两个模型分别要跑一遍，配置放一起方便遍历
MODEL_CONFIGS = [
    {"name": "task_aware", "ckpt": TASK_AWARE_CKPT},
    {"name": "low_bpp",    "ckpt": LOW_BPP_CKPT},
]


# =====================================================================
# 数据集：不做随机增强（生成数据对不需要翻转/裁剪），保证和原图严格对应
# =====================================================================
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.Lambda(lambda img: img.convert("RGB") if img.mode != "RGB" else img),
    transforms.ToTensor(),
])


class DFireImageOnly(Dataset):
    """与之前 DFireDataset 一致的目录假设，这里只需要图片，不需要标注"""
    def __init__(self, root, split):
        img_dir = os.path.join(root, split, "images")
        self.paths = sorted(
            glob.glob(os.path.join(img_dir, "*.jpg"))
            + glob.glob(os.path.join(img_dir, "*.png"))
        )
        if not self.paths:
            raise FileNotFoundError(f"在 {img_dir} 下找不到图片")

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        path = self.paths[idx]
        img = Image.open(path).convert("RGB")
        img = transform(img)
        stem = os.path.splitext(os.path.basename(path))[0]
        return img, stem


def maybe_subset(dataset, max_samples):
    if max_samples is None or max_samples >= len(dataset):
        return dataset
    return Subset(dataset, list(range(max_samples)))


print(f"Device: {DEVICE}", flush=True)

print("加载数据集...", flush=True)
train_db = maybe_subset(DFireImageOnly(DATA_ROOT, "train"), MAX_TRAIN_SAMPLES)
val_db   = maybe_subset(DFireImageOnly(DATA_ROOT, "test"),  MAX_VAL_SAMPLES)

train_loader = DataLoader(train_db, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
val_loader   = DataLoader(val_db,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print(f"Train: {len(train_db)} 张 | Val: {len(val_db)} 张", flush=True)


# =====================================================================
# 核心函数：用给定压缩模型对一个 loader 跑重建，保存 (orig, recon) 配对
# =====================================================================
@torch.no_grad()
def generate_pairs(model, loader, out_dir, split_name):
    orig_dir  = os.path.join(out_dir, split_name, "orig")
    recon_dir = os.path.join(out_dir, split_name, "recon")
    os.makedirs(orig_dir, exist_ok=True)
    os.makedirs(recon_dir, exist_ok=True)

    model.eval()
    saved = 0
    for imgs, stems in tqdm(loader, desc=f"生成 {split_name} 数据对"):
        imgs = imgs.to(DEVICE)
        out = model(imgs)
        x_hat = out["x_hat"].clamp(0, 1)

        for i, stem in enumerate(stems):
            # 原图和重建图用同名文件保存在不同目录，方便 RDDM 训练时按文件名配对
            save_image(imgs[i].cpu(),  os.path.join(orig_dir,  f"{stem}.png"))
            save_image(x_hat[i].cpu(), os.path.join(recon_dir, f"{stem}.png"))
            saved += 1

    print(f"  已保存 {saved} 对图片到 {out_dir}/{split_name}/", flush=True)
    return saved


# =====================================================================
# 主流程：对两个压缩模型分别生成 train/val 数据对
# =====================================================================
for cfg in MODEL_CONFIGS:
    name = cfg["name"]
    ckpt_path = cfg["ckpt"]

    print(f"\n{'=' * 60}")
    print(f"模型: {name}  ({ckpt_path})")
    print(f"{'=' * 60}", flush=True)

    if not os.path.exists(ckpt_path):
        print(f"  [跳过] 找不到权重文件: {ckpt_path}", flush=True)
        continue

    model = Cheng2020Anchor(N=128).to(DEVICE)
    ckpt = torch.load(ckpt_path, map_location=DEVICE)
    model.load_state_dict(ckpt["model"])
    print(f"  已加载权重 (PSNR={ckpt['val_psnr']:.2f} dB, "
          f"bpp={ckpt.get('val_bpp', float('nan')):.4f})", flush=True)

    out_dir = os.path.join(OUT_ROOT, name)

    generate_pairs(model, train_loader, out_dir, "train")
    generate_pairs(model, val_loader,   out_dir, "val")

    del model
    torch.cuda.empty_cache()

print("\n全部完成。数据对目录结构：")
for cfg in MODEL_CONFIGS:
    name = cfg["name"]
    base = os.path.join(OUT_ROOT, name)
    if os.path.exists(base):
        for split in ["train", "val"]:
            orig_dir = os.path.join(base, split, "orig")
            if os.path.exists(orig_dir):
                print(f"  {base}/{split}/  — {len(os.listdir(orig_dir))} 对")

print(f"\n下一步：用 {OUT_ROOT}/<task_aware|low_bpp>/train/{{orig,recon}} "
      f"替换 RDDM 仓库（nachifur/RDDM）的数据接口，"
      f"训练目标是从 recon 预测残差 r = orig - recon。")

In [ ]:
"""
简化版残差增强网络 —— RDDM 思路的单步实现

不是完整复现 RDDM 论文（不含多步扩散采样、不含噪声扩散分支），
只保留核心思想：用一个网络直接学习 r = orig - recon，
训练目标：L1(预测残差, 真实残差)
推理：enhanced = recon + 预测残差

这样做的好处：训练快、参数少、不依赖原仓库复杂的参数耦合，
代价：失去了扩散模型多步迭代精化和生成多样性的能力，
本质上等价于一个轻量级图像增强/去伪影网络。

数据来源：generate_rddm_pairs.py 生成的 (orig, recon) 配对
  /content/rddm_data/<task_aware|low_bpp>/train/{orig,recon}
  /content/rddm_data/<task_aware|low_bpp>/val/{orig,recon}
"""

import os
import glob
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(iterable, **kwargs):
        return iterable

# =====================================================================
# 配置
# =====================================================================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 选择对哪个压缩模型的重建图做残差增强：
# "low_bpp" 问题更严重(bpp≈0.18)，"task_aware" 损失较小(bpp≈0.42)
# 建议先用 low_bpp，因为这是检测精度损失最大、RDDM最有价值的场景
TARGET = "low_bpp"   # 改成 "task_aware" 可以训另一套

DATA_ROOT  = f"/content/rddm_data/{TARGET}"
IMG_SIZE   = 384
BATCH_SIZE = 8        # UNet 比压缩模型更吃显存，T4 上从 8 起步
NUM_WORKERS = 0
EPOCHS     = 30
EARLY_STOP_PATIENCE = 6

LR = 1e-4
GRAD_CLIP_NORM = 1.0

save_dir = f"outputs_residual_enhance_{TARGET}"
os.makedirs(save_dir, exist_ok=True)

history = {"train_loss": [], "val_loss": [], "train_psnr": [], "val_psnr": []}


# =====================================================================
# 数据集：直接读 generate_rddm_pairs.py 产出的 orig/recon 配对
# =====================================================================
to_tensor = transforms.ToTensor()  # 图片已经是 384x384 PNG，不需要再 Resize


class ResidualPairDataset(Dataset):
    def __init__(self, root, split):
        self.orig_dir  = os.path.join(root, split, "orig")
        self.recon_dir = os.path.join(root, split, "recon")
        self.stems = sorted(
            os.path.splitext(os.path.basename(p))[0]
            for p in glob.glob(os.path.join(self.orig_dir, "*.png"))
        )
        if not self.stems:
            raise FileNotFoundError(f"在 {self.orig_dir} 下找不到图片，请先运行 generate_rddm_pairs.py")

    def __len__(self):
        return len(self.stems)

    def __getitem__(self, idx):
        stem = self.stems[idx]
        orig  = to_tensor(Image.open(os.path.join(self.orig_dir,  f"{stem}.png")).convert("RGB"))
        recon = to_tensor(Image.open(os.path.join(self.recon_dir, f"{stem}.png")).convert("RGB"))
        return orig, recon


@torch.no_grad()
def batch_psnr(recon, target):
    mse = (recon - target).pow(2).mean(dim=(1, 2, 3))
    return (10 * torch.log10(1.0 / mse.clamp(min=1e-8))).mean().item()


print(f"Device: {DEVICE}", flush=True)
print(f"目标压缩模型: {TARGET}", flush=True)

train_db = ResidualPairDataset(DATA_ROOT, "train")
val_db   = ResidualPairDataset(DATA_ROOT, "val")

train_loader = DataLoader(train_db, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"))
val_loader   = DataLoader(val_db,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"))

print(f"Train: {len(train_db)} 对 | Val: {len(val_db)} 对", flush=True)


# =====================================================================
# 模型：轻量 UNet，输入 recon，输出残差预测
# 比压缩模型的 encoder/decoder 更浅，因为任务更简单（只是局部纹理修复，
# 不需要重新学习整个压缩/解压映射）
# =====================================================================
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, 1, 1),
            nn.GroupNorm(8, out_ch),
            nn.SiLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, 1, 1),
            nn.GroupNorm(8, out_ch),
            nn.SiLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class ResidualUNet(nn.Module):
    """
    标准 UNet 结构：3 次下采样 + 3 次上采样 + skip connection。
    输入: recon [B,3,H,W]
    输出: 预测残差 r_hat [B,3,H,W]，范围不限制（残差可正可负）
    """
    def __init__(self, base_ch=32):
        super().__init__()
        # Encoder
        self.enc1 = ConvBlock(3, base_ch)            # 384
        self.down1 = nn.Conv2d(base_ch, base_ch, 4, 2, 1)       # 384→192

        self.enc2 = ConvBlock(base_ch, base_ch * 2)
        self.down2 = nn.Conv2d(base_ch * 2, base_ch * 2, 4, 2, 1)  # 192→96

        self.enc3 = ConvBlock(base_ch * 2, base_ch * 4)
        self.down3 = nn.Conv2d(base_ch * 4, base_ch * 4, 4, 2, 1)  # 96→48

        # Bottleneck
        self.bottleneck = ConvBlock(base_ch * 4, base_ch * 8)

        # Decoder（带 skip connection，所以 in_ch 是两倍）
        self.up3 = nn.ConvTranspose2d(base_ch * 8, base_ch * 4, 4, 2, 1)  # 48→96
        self.dec3 = ConvBlock(base_ch * 8, base_ch * 4)

        self.up2 = nn.ConvTranspose2d(base_ch * 4, base_ch * 2, 4, 2, 1)  # 96→192
        self.dec2 = ConvBlock(base_ch * 4, base_ch * 2)

        self.up1 = nn.ConvTranspose2d(base_ch * 2, base_ch, 4, 2, 1)      # 192→384
        self.dec1 = ConvBlock(base_ch * 2, base_ch)

        # 输出层：不加激活函数，残差需要能取负值
        self.out_conv = nn.Conv2d(base_ch, 3, 3, 1, 1)
        # 输出层权重初始化为接近0，训练初期网络接近恒等映射（enhanced≈recon），
        # 这样不会一开始就把图像搞乱，是残差学习的标准技巧
        nn.init.zeros_(self.out_conv.weight)
        nn.init.zeros_(self.out_conv.bias)

    def forward(self, x):
        e1 = self.enc1(x)
        d1 = self.down1(e1)

        e2 = self.enc2(d1)
        d2 = self.down2(e2)

        e3 = self.enc3(d2)
        d3 = self.down3(e3)

        b = self.bottleneck(d3)

        u3 = self.up3(b)
        u3 = self.dec3(torch.cat([u3, e3], dim=1))

        u2 = self.up2(u3)
        u2 = self.dec2(torch.cat([u2, e2], dim=1))

        u1 = self.up1(u2)
        u1 = self.dec1(torch.cat([u1, e1], dim=1))

        r_hat = self.out_conv(u1)
        return r_hat


print("Building model...", flush=True)
model = ResidualUNet(base_ch=32).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"模型参数量: {n_params / 1e6:.1f}M", flush=True)

optimizer = optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=3, min_lr=1e-7
)

# =====================================================================
# 训练循环
# =====================================================================
best_psnr = 0.0
epochs_no_improve = 0

for epoch in range(EPOCHS):
    t_epoch = time.time()

    # ---------- Train ----------
    model.train()
    total_loss = total_psnr = 0.0
    n_batches = 0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [train]", leave=False)
    for orig, recon in pbar:
        orig, recon = orig.to(DEVICE), recon.to(DEVICE)
        target_residual = orig - recon

        optimizer.zero_grad(set_to_none=True)
        pred_residual = model(recon)

        loss = F.l1_loss(pred_residual, target_residual)
        loss.backward()
        if GRAD_CLIP_NORM:
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
        optimizer.step()

        with torch.no_grad():
            enhanced = (recon + pred_residual).clamp(0, 1)
            psnr = batch_psnr(enhanced, orig)

        total_loss += loss.item()
        total_psnr += psnr
        n_batches += 1
        pbar.set_postfix(loss=f"{loss.item():.4f}", psnr=f"{psnr:.2f}")

    avg_train_loss = total_loss / n_batches
    avg_train_psnr = total_psnr / n_batches

    # ---------- Validation ----------
    model.eval()
    val_loss = val_psnr = 0.0
    val_psnr_baseline = 0.0   # 不经过增强网络，recon本身相对orig的PSNR，作为对照
    n_val = 0

    with torch.no_grad():
        for orig, recon in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [val]", leave=False):
            orig, recon = orig.to(DEVICE), recon.to(DEVICE)
            target_residual = orig - recon

            pred_residual = model(recon)
            loss = F.l1_loss(pred_residual, target_residual)

            enhanced = (recon + pred_residual).clamp(0, 1)

            val_loss += loss.item()
            val_psnr += batch_psnr(enhanced, orig)
            val_psnr_baseline += batch_psnr(recon, orig)
            n_val += 1

    avg_val_loss = val_loss / n_val
    avg_val_psnr = val_psnr / n_val
    avg_val_psnr_baseline = val_psnr_baseline / n_val
    elapsed = time.time() - t_epoch

    history["train_loss"].append(avg_train_loss)
    history["val_loss"].append(avg_val_loss)
    history["train_psnr"].append(avg_train_psnr)
    history["val_psnr"].append(avg_val_psnr)

    scheduler.step(avg_val_psnr)

    print(
        f"Epoch {epoch+1}/{EPOCHS} ({elapsed:.1f}s) | "
        f"Train Loss {avg_train_loss:.4f} | Val Loss {avg_val_loss:.4f} | "
        f"Train PSNR {avg_train_psnr:.2f} | Val PSNR(enhanced) {avg_val_psnr:.2f} dB | "
        f"Val PSNR(recon,baseline) {avg_val_psnr_baseline:.2f} dB | "
        f"提升 {avg_val_psnr - avg_val_psnr_baseline:+.2f} dB",
        flush=True,
    )

    if avg_val_psnr > best_psnr:
        best_psnr = avg_val_psnr
        epochs_no_improve = 0
        torch.save(
            {
                "epoch": epoch + 1,
                "model": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "val_psnr": avg_val_psnr,
                "val_psnr_baseline": avg_val_psnr_baseline,
                "target": TARGET,
            },
            f"{save_dir}/best_model.pth",
        )
        print(f"  → 保存最优模型 PSNR={best_psnr:.2f} dB "
              f"(相比未增强提升 {best_psnr - avg_val_psnr_baseline:+.2f} dB)", flush=True)
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= EARLY_STOP_PATIENCE:
            print(f"Early stop: Val PSNR 连续 {EARLY_STOP_PATIENCE} epoch 未超过 {best_psnr:.2f} dB", flush=True)
            break

# =====================================================================
# 绘图
# =====================================================================
epochs_ran = range(1, len(history["train_loss"]) + 1)

plt.figure()
plt.plot(epochs_ran, history["train_loss"], label="Train L1 Loss")
plt.plot(epochs_ran, history["val_loss"],   label="Val L1 Loss")
plt.xlabel("Epoch"); plt.ylabel("L1 Loss")
plt.title(f"Residual Loss ({TARGET})"); plt.legend(); plt.grid()
plt.savefig(f"{save_dir}/loss_curve.png"); plt.show()

plt.figure()
plt.plot(epochs_ran, history["train_psnr"], label="Train PSNR (enhanced)")
plt.plot(epochs_ran, history["val_psnr"],   label="Val PSNR (enhanced)")
plt.axhline(y=avg_val_psnr_baseline, color="gray", linestyle="--",
            label=f"Baseline (no enhance) = {avg_val_psnr_baseline:.2f}dB")
plt.xlabel("Epoch"); plt.ylabel("PSNR (dB)")
plt.title(f"PSNR Improvement ({TARGET})"); plt.legend(); plt.grid()
plt.savefig(f"{save_dir}/psnr_curve.png"); plt.show()

print(f"\n训练完成。")
print(f"  未增强 baseline PSNR : {avg_val_psnr_baseline:.2f} dB")
print(f"  增强后最优 PSNR      : {best_psnr:.2f} dB")
print(f"  提升                : {best_psnr - avg_val_psnr_baseline:+.2f} dB")